# Logistic Regression from Scratch with PyTorch

binary image classifier for **bee vs. ant** images

In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Selected device: {device}")

PyTorch version: 2.14.1
Selected device: cpu


In [2]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

data_dir = Path("data")
dataset_dir = data_dir / "hymenoptera_data"
archive_path = data_dir / "hymenoptera_data.zip"
dataset_url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"

data_dir.mkdir(exist_ok=True)
if not dataset_dir.exists():
    if not archive_path.exists():
        urlretrieve(dataset_url, archive_path)

    with ZipFile(archive_path) as archive:
        archive.extractall(data_dir)

for split in ("train", "val"):
    print(f"\n{split}:")
    for label in ("ants", "bees"):
        image_dir = dataset_dir / split / label
        image_count = sum(
            path.is_file() for path in image_dir.iterdir()
        )
        print(f"  {label}: {image_count} images")



train:
  ants: 124 images
  bees: 121 images

val:
  ants: 70 images
  bees: 83 images


In [3]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [4]:
stats_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

stats_dataset = datasets.ImageFolder(
    dataset_dir / "train",
    transform=stats_transform,
)
stats_loader = DataLoader(
    stats_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
)

channel_sum = torch.zeros(3, dtype=torch.float64)
channel_sum_squared = torch.zeros(3, dtype=torch.float64)
pixel_count = 0

In [5]:

for images, _ in stats_loader:
    images = images.to(torch.float64)
    channel_sum += images.sum(dim=(0, 2, 3))
    channel_sum_squared += (images ** 2).sum(dim=(0, 2, 3))
    pixel_count += images.shape[0] * images.shape[2] * images.shape[3]

image_mean = channel_sum / pixel_count
image_std = (channel_sum_squared / pixel_count - image_mean ** 2).sqrt()

image_mean = image_mean.tolist()
image_std = image_std.tolist()

print("Training-set mean:", image_mean)
print("Training-set std:", image_std)

Training-set mean: [0.5198643883272248, 0.47025237820269467, 0.34418210152745055]
Training-set std: [0.27000941419063357, 0.2535465938093949, 0.2807125881120736]


In [6]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(image_mean, image_std),
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(image_mean, image_std),
])

train_dataset = datasets.ImageFolder(
    dataset_dir / "train",
    transform=train_transforms,
)
val_dataset = datasets.ImageFolder(
    dataset_dir / "val",
    transform=val_transforms,
)


In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
)

print("Class mapping:", train_dataset.class_to_idx)

images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("First batch labels:", labels)

Class mapping: {'ants': 0, 'bees': 1}
Image batch shape: torch.Size([16, 3, 224, 224])
Label batch shape: torch.Size([16])
First batch labels: tensor([1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1])


In [8]:
import torch.nn as nn

num_features = 3 * 224 * 224

In [9]:
class LogisticRegression(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.weights = nn.Parameter(torch.zeros(num_features, 1))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, images):
        flattened_images = images.reshape(images.shape[0], -1)
        logits = flattened_images @ self.weights + self.bias
        return logits.squeeze(1)


model = LogisticRegression(num_features).to(device)

images, labels = next(iter(train_loader))
images = images.to(device)

logits = model(images)
probabilities = torch.sigmoid(logits)

print("Logits shape:", logits.shape)
print("First probabilities:", probabilities[:5])

Logits shape: torch.Size([16])
First probabilities: tensor([0.5000, 0.5000, 0.5000, 0.5000, 0.5000], grad_fn=<SliceBackward0>)


In [10]:
loss_fn = nn.BCEWithLogitsLoss()

images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device).float()

with torch.no_grad():
    logits = model(images)
    loss = loss_fn(logits, labels)

    predictions = (logits >= 0).float()
    accuracy = (predictions == labels).float().mean() * 100

print(f"Initial loss: {loss.item():.4f}")
print(f"Initial accuracy: {accuracy.item():.2f}%")

Initial loss: 0.6931
Initial accuracy: 68.75%


In [11]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    train_loss_sum = 0.0
    train_correct = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device).float()

        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * labels.size(0)
        train_correct += ((logits >= 0) == labels.bool()).sum().item()

    train_loss = train_loss_sum / len(train_dataset)
    train_accuracy = train_correct / len(train_dataset) * 100

    model.eval()
    val_loss_sum = 0.0
    val_correct = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device).float()

            logits = model(images)
            loss = loss_fn(logits, labels)

            val_loss_sum += loss.item() * labels.size(0)
            val_correct += ((logits >= 0) == labels.bool()).sum().item()

    val_loss = val_loss_sum / len(val_dataset)
    val_accuracy = val_correct / len(val_dataset) * 100

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"train loss: {train_loss:.4f}, "
        f"train accuracy: {train_accuracy:.2f}% | "
        f"val loss: {val_loss:.4f}, "
        f"val accuracy: {val_accuracy:.2f}%"
    )

Epoch 1/5 | train loss: 11.6844, train accuracy: 52.46% | val loss: 11.6253, val accuracy: 51.63%
Epoch 2/5 | train loss: 10.1720, train accuracy: 51.23% | val loss: 5.0043, val accuracy: 66.67%
Epoch 3/5 | train loss: 10.2962, train accuracy: 57.79% | val loss: 9.3000, val accuracy: 52.29%
Epoch 4/5 | train loss: 11.5691, train accuracy: 56.97% | val loss: 13.0553, val accuracy: 50.98%
Epoch 5/5 | train loss: 11.0150, train accuracy: 60.25% | val loss: 9.1748, val accuracy: 55.56%
